1. Importar bibliotecas

In [ ]:
import os
from pathlib import Path
from time import perf_counter

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.engine import URL

2. Configurar o dataset

In [ ]:
DATASET_URL = (
    "https://github.com/DataTalksClub/nyc-tlc-data/releases/"
    "download/yellow/yellow_tripdata_2021-01.csv.gz"
)

TABLE_NAME = "yellow_taxi_data_notebook"
READ_CHUNK_SIZE = 100_000
WRITE_CHUNK_SIZE = 10_000

3. Explorar os primeiros 100 registos

In [ ]:
df_sample = pd.read_csv(
    DATASET_URL,
    nrows=100,
)

display(df_sample.head())

print("Shape:", df_sample.shape)
print("\nData types:")
print(df_sample.dtypes)

print("\nNull values:")
print(df_sample.isna().sum())

4. Definir os tipos de dados

In [ ]:
DTYPE = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64",
}

PARSE_DATES = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]

5. Validar os tipos definidos

In [ ]:
df_sample = pd.read_csv(
    DATASET_URL,
    nrows=100,
    dtype=DTYPE,
    parse_dates=PARSE_DATES,
)

print("Data types:")
print(df_sample.dtypes)

print("\nPickup date range:")
print(df_sample["tpep_pickup_datetime"].min())
print(df_sample["tpep_pickup_datetime"].max())

print("\nNull values:")
print(df_sample.isna().sum())

6. Carregar as variáveis de ambiente

In [ ]:
ENV_FILE = Path.cwd().parent / "03-postgresql" / ".env"

if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Environment file not found: {ENV_FILE}")

load_dotenv(ENV_FILE)

print("Environment configuration loaded")

7. Estabelecer ligação ao PostgreSQL

In [ ]:
database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    host="127.0.0.1",
    port=5432,
    database=os.environ["POSTGRES_DB"],
)

engine = create_engine(
    database_url,
    connect_args={"connect_timeout": 5},
)

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT current_database(), current_user")
    )

    print("PostgreSQL connection:", result.fetchone())

8. Analisar o schema SQL

In [ ]:
schema = pd.io.sql.get_schema(
    df_sample,
    name=TABLE_NAME,
    con=engine,
)

print(schema)

9. Preparar o leitor de chunks

In [ ]:
chunks = pd.read_csv(
    DATASET_URL,
    dtype=DTYPE,
    parse_dates=PARSE_DATES,
    chunksize=READ_CHUNK_SIZE,
)

print("Reader type:", type(chunks))

10. Carregar os dados no PostgreSQL - Esta célula executa a ingestão completa. Substitui a tabela yellow_taxi_data_notebook se ela já existir. Não modifica yellow_taxi_data nem yellow_taxi_data_pipeline.

In [ ]:
total_rows = 0
start_total = perf_counter()

for batch_number, df_chunk in enumerate(chunks, start=1):
    start_batch = perf_counter()

    df_chunk.to_sql(
        name=TABLE_NAME,
        con=engine,
        if_exists="replace" if batch_number == 1 else "append",
        index=False,
        chunksize=WRITE_CHUNK_SIZE,
    )

    total_rows += len(df_chunk)

    elapsed = perf_counter() - start_batch

    print(
        f"Batch {batch_number}: "
        f"{len(df_chunk):,} rows | "
        f"{elapsed:.2f}s"
    )

print(f"\nTotal rows ingested: {total_rows:,}")
print(f"Total execution time: {perf_counter() - start_total:.2f}s")

11. Confirmar a tabela e o schema PostgreSQL

In [ ]:
inspector = inspect(engine)

print("Table exists:", inspector.has_table(TABLE_NAME))

for column in inspector.get_columns(TABLE_NAME):
    print(column["name"], ":", column["type"])

12. Validar a qualidade dos dados

In [ ]:
query = text(f"""
SELECT
    COUNT(*) AS total_trips,
    MIN(tpep_pickup_datetime) AS first_pickup,
    MAX(tpep_pickup_datetime) AS last_pickup,
    COUNT(*) FILTER (
        WHERE tpep_pickup_datetime IS NULL
    ) AS null_pickup_dates,
    COUNT(*) FILTER (
        WHERE trip_distance < 0
    ) AS negative_distances
FROM {TABLE_NAME}
""")

validation = pd.read_sql(query, con=engine)

display(validation)

13. Verificar anomalias temporais

In [ ]:
query = text(f"""
SELECT
    COUNT(*) FILTER (
        WHERE tpep_pickup_datetime < '2021-01-01'
    ) AS trips_before_january,

    COUNT(*) FILTER (
        WHERE tpep_pickup_datetime >= '2021-02-01'
    ) AS trips_after_january,

    COUNT(*) FILTER (
        WHERE tpep_pickup_datetime >= '2021-01-01'
          AND tpep_pickup_datetime < '2021-02-01'
    ) AS trips_in_january
FROM {TABLE_NAME}
""")

date_validation = pd.read_sql(query, con=engine)

display(date_validation)

14. Comparar o notebook com o pipeline Python

In [ ]:
query = """
SELECT
    (SELECT COUNT(*) FROM yellow_taxi_data_notebook) AS notebook_rows,
    (SELECT COUNT(*) FROM yellow_taxi_data_pipeline) AS pipeline_rows
"""

comparison = pd.read_sql(query, con=engine)

display(comparison)

15. Libertar recursos

In [ ]:
engine.dispose()

print("Database connections released")